In [1]:
# 05d_final_eval_sensitivity.ipynb
# =============================================================================
# Notebook 05d — Final hallucination/governance eval with PRINCIPLED range policy
#
# PRIMARY policy (pre-registered here, chosen on design logic NOT on results):
#   lower bound : may widen to data 5th percentile (reachability)
#   upper bound : KEEP hard-rule ceiling, never widen (upper bounds ARE the
#                 safety constraints: carb/sugar<=baseline for conflict
#                 prevention, sodium<=0.9c, energy<=baseline)
#
# SENSITIVITY policies (reported for robustness, NOT for cherry-picking):
#   strict_both : neither bound widened
#   widen_both  : both bounds widened (the buggy original; kept to show its effect)
#
# Also logs, per case, WHY a condition yields <4 CFs:
#   class0_reached and n_cf, to separate "infeasible" from "few candidates".
#
# Output: ../results/tables/final_eval_percf.csv
#         ../results/tables/final_eval_summary.csv
#         ../results/tables/final_eval_feasibility.csv
# =============================================================================

# %%
import json, joblib, warnings
import numpy as np, pandas as pd
import dice_ml
warnings.filterwarnings('ignore')

EXT_ENERGY_FLOOR = 800.0
CHANGE_TOL = 1.0

agent_config = joblib.load('../results/tables/agent_config.pkl')
df_final     = joblib.load('../results/tables/df_final.pkl')
X_FEATURES      = agent_config['X_features']
VARY_FEATURES   = agent_config['vary_features']
TARGET_COL      = agent_config['target_col']
AGEGROUP_CONFIG = agent_config['agegroup_config']
models = {g: joblib.load(f'../results/tables/model_{g}.pkl') for g in AGEGROUP_CONFIG}
with open('../results/tables/guardrail_ranges_v2.json', encoding='utf-8') as f:
    GR = json.load(f)

# %%
# ## Range builders — three policies
def build_primary(ranges, df_ref, features):
    """PRIMARY: widen LOWER bound to data p5; KEEP upper bound (hard ceiling)."""
    sr = {}
    for f in features:
        pair = ranges.get(f)
        if pair is None: continue
        lo, hi = float(pair[0]), float(pair[1])
        dlo = float(df_ref[f].quantile(0.05))
        lo = max(min(lo, dlo), 0.0)      # lower may widen down
        sr[f] = [round(lo,4), round(hi,4)]   # upper untouched
    return sr

def build_strict(ranges, features):
    """SENSITIVITY: neither bound widened."""
    return {f: [round(float(p[0]),4), round(float(p[1]),4)]
            for f,p in ranges.items() if p is not None}

def build_widen(ranges, df_ref, features):
    """SENSITIVITY: both bounds widened (original buggy behaviour)."""
    sr = {}
    for f in features:
        pair = ranges.get(f)
        if pair is None: continue
        lo, hi = float(pair[0]), float(pair[1])
        dlo=float(df_ref[f].quantile(0.05)); dhi=float(df_ref[f].quantile(0.95))
        sr[f] = [round(max(min(lo,dlo),0.0),4), round(max(hi,dhi),4)]
    return sr

# %%
# ## Violation detectors (external)
def v_energy(cf, orig):
    v=float(cf.get('Energy_kcal',orig.get('Energy_kcal',0)))
    return int(v < EXT_ENERGY_FLOOR)

def v_conflict(cf, orig, tol=CHANGE_TOL):
    c=float(orig.get('Sodium_mg',0)); v=float(cf.get('Sodium_mg',c))
    if not (c>0 and (c-v)/c*100>tol): return 0
    cc=float(orig.get('Carb_g',0)); vc=float(cf.get('Carb_g',cc))
    cs=float(orig.get('Sugar_g',0)); vs=float(cf.get('Sugar_g',cs))
    up_c=(vc-cc)/cc*100>tol if cc>0 else False
    up_s=(vs-cs)/cs*100>tol if cs>0 else False
    return int(up_c or up_s)

def v_bmiwt(cf, orig, tol=CHANGE_TOL):
    def chg(k):
        o=float(orig.get(k,0)); v=float(cf.get(k,o)); return (v-o)/o*100 if o else 0
    b,w=chg('BMI'),chg('Weight')
    if abs(b)<=tol or abs(w)<=tol: return 0
    return int(np.sign(b)!=np.sign(w))

def clip_to(cf, ranges):
    out=dict(cf)
    for f,p in ranges.items():
        if f in out and p is not None:
            out[f]=float(np.clip(float(out[f]),float(p[0]),float(p[1])))
    return out

def gen_cfs(exp, query, permitted=None, n=4):
    kw=dict(total_CFs=n, desired_class=0, features_to_vary=VARY_FEATURES,
            proximity_weight=0.2, sparsity_weight=0.1)
    if permitted: kw['permitted_range']=permitted
    try:
        cf=exp.generate_counterfactuals(query, **kw)
        return cf.cf_examples_list[0].final_cfs_df.to_dict('records') if cf else []
    except Exception:
        return []

# %%
records=[]; feas=[]
POLICIES=['primary','strict','widen']

for case_key,g in GR.items():
    grp=g['group']; mdl=models.get(grp)
    if mdl is None: continue
    orig=g['patient_profile']; cfg=AGEGROUP_CONFIG[grp]
    age=0.0 if cfg['age_min']<60 else 1.0
    dref=df_final[(df_final['AgeGroup']==age)&(df_final['Sex']==cfg['sex_code'])].copy()
    query=pd.DataFrame([orig])[X_FEATURES]
    d=dice_ml.Data(dataframe=df_final.copy().astype(float)[X_FEATURES+[TARGET_COL]],
                   continuous_features=X_FEATURES, outcome_name=TARGET_COL)
    m=dice_ml.Model(model=mdl, backend='sklearn')
    exp=dice_ml.Dice(d,m,method='genetic')

    hard=g['hardrule_ranges']
    perms={
        'primary': build_primary(g['final_ranges'], dref, X_FEATURES),
        'strict':  build_strict(g['final_ranges'], X_FEATURES),
        'widen':   build_widen(g['final_ranges'], dref, X_FEATURES),
    }

    # Pure DiCE baseline
    pure=gen_cfs(exp,query,None)
    for cf in pure:
        records.append({'CaseKey':case_key,'Group':grp,'Policy':'-','Stage':'PureDiCE',
                        'V_energy':v_energy(cf,orig),'V_conflict':v_conflict(cf,orig),
                        'V_bmiwt':v_bmiwt(cf,orig)})
    feas.append({'CaseKey':case_key,'Policy':'-','Stage':'PureDiCE','n_CF':len(pure)})

    for pol in POLICIES:
        soft=gen_cfs(exp,query,perms[pol])
        feas.append({'CaseKey':case_key,'Policy':pol,'Stage':'soft','n_CF':len(soft)})
        for cf in soft:
            records.append({'CaseKey':case_key,'Group':grp,'Policy':pol,'Stage':'soft',
                            'V_energy':v_energy(cf,orig),'V_conflict':v_conflict(cf,orig),
                            'V_bmiwt':v_bmiwt(cf,orig)})
        for cf in soft:
            cfh=clip_to(cf,hard)
            records.append({'CaseKey':case_key,'Group':grp,'Policy':pol,'Stage':'hard',
                            'V_energy':v_energy(cfh,orig),'V_conflict':v_conflict(cfh,orig),
                            'V_bmiwt':v_bmiwt(cfh,orig)})
    print(f"  [{case_key}] done")

rec=pd.DataFrame(records); rec.to_csv('../results/tables/final_eval_percf.csv',index=False,encoding='utf-8-sig')
fe=pd.DataFrame(feas); fe.to_csv('../results/tables/final_eval_feasibility.csv',index=False,encoding='utf-8-sig')

# %%
print("\n=== Violation rate (%) by policy x stage ===")
def block(pol):
    sub=rec[(rec['Policy']==pol)|(rec['Stage']=='PureDiCE')]
    s=(sub.groupby('Stage')[['V_energy','V_conflict','V_bmiwt']].mean()*100).round(1)
    s['n_CF']=sub.groupby('Stage').size()
    return s.reindex([x for x in ['PureDiCE','soft','hard'] if x in s.index])

for pol in POLICIES:
    print(f"\n--- POLICY = {pol}{'  (PRIMARY)' if pol=='primary' else '  (sensitivity)'} ---")
    print(block(pol).to_string())

print("\n=== Feasibility: mean CFs generated per case ===")
print(fe.groupby(['Policy','Stage'])['n_CF'].agg(['mean','min']).round(2).to_string())

# %%
summary=[]
for pol in POLICIES:
    b=block(pol)
    for stage in b.index:
        summary.append({'Policy':pol,'Stage':stage,
                        'V_energy_%':b.loc[stage,'V_energy'],
                        'V_conflict_%':b.loc[stage,'V_conflict'],
                        'V_bmiwt_%':b.loc[stage,'V_bmiwt'],
                        'n_CF':int(b.loc[stage,'n_CF'])})
pd.DataFrame(summary).to_csv('../results/tables/final_eval_summary.csv',index=False,encoding='utf-8-sig')
print("\n>>> Saved final_eval_summary.csv")

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.86it/s]


  [MiddleAged_Male_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.11it/s]


  [MiddleAged_Male_case2] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.37it/s]


...

  [Older_Female_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.11it/s]


  [Older_Female_case2] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.27it/s]

  [Older_Female_case3] done

=== Violation rate (%) by policy x stage ===

--- POLICY = primary  (PRIMARY) ---
          V_energy  V_conflict  V_bmiwt  n_CF
Stage                                        
PureDiCE       4.2        33.3      8.3    48
soft           8.5         0.0      0.0    47
hard           0.0         0.0      0.0    47

--- POLICY = strict  (sensitivity) ---
          V_energy  V_conflict  V_bmiwt  n_CF
Stage                                        
PureDiCE       4.2        33.3      8.3    48
soft           0.0         0.0      0.0    38
hard           0.0         0.0      0.0    38

--- POLICY = widen  (sensitivity) ---
          V_energy  V_conflict  V_bmiwt  n_CF
Stage                                        
PureDiCE       4.2        33.3      8.3    48
soft           8.5        46.8      6.4    47
hard           0.0         0.0      0.0    47

=== Feasibility: mean CFs generated per case ===
                  mean  min
Policy  Stage              
-       PureDi